## Statistics, Publication Tables & Figures — Experiments 6, 7, 8, 10, 11, 12

**Notebook 3 of 3.** This notebook is **evaluation/aggregation-only, no retraining and no fresh
model inference** — it consumes the artifacts already produced by `04a` (per-model results
pickles: raw `y_true`/`y_pred`/`y_proba`, per-seed metrics, training/inference cost) and `04b`
(cross-dataset tables, language-breakdown table, ablation table, error-analysis CSVs, dataset-
transfer stats). If an expected artifact is missing, cells fail loudly with the exact path so you
can re-run the right upstream cell instead of silently producing partial results.

Covers:
- **Experiment 6** — Threshold optimization (Unified test set)
- **Experiment 7** — Statistical significance (pairwise McNemar + multi-seed variance + bootstrap CIs)
- **Experiment 8** — Error-analysis writeup (quantitative synthesis of `04b`'s error CSVs)
- **Experiment 10** — Efficiency (model size, params, train/infer time, throughput)
- **Experiment 11** — Publication tables (camera-ready .tex/.md/.csv)
- **Experiment 12** — Remaining publication figures


## 1. Setup & Install

In [ ]:
!pip install -q scikit-learn matplotlib seaborn scipy statsmodels openpyxl
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%matplotlib inline
import os, json, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_fscore_support, precision_recall_curve, roc_curve,
    accuracy_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
)
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Imports OK.")


## 2. Config — paths (same layout as 05a/04b)

In [ ]:
SEED = 42

DATASETS_BASE = "/content/drive/MyDrive/Datasets"
LAYER1_DIR    = f"{DATASETS_BASE}/layer1"
OUTPUT_DIR    = f"{LAYER1_DIR}/outputs"

# 04a outputs
RESULTS_DIR = f"{OUTPUT_DIR}/results"
PREDS_DIR   = f"{OUTPUT_DIR}/predictions"     
# 04b outputs
B_OUT        = f"{OUTPUT_DIR}/04b"
CROSS_DIR    = f"{B_OUT}/cross_dataset"
LANG_DIR     = f"{B_OUT}/language_analysis"
ABLATION_DIR = f"{B_OUT}/ablation"
ERROR_DIR    = f"{B_OUT}/error_analysis"
SUMMARY_DIR  = f"{B_OUT}/summary"

# 04c outputs (this notebook)
C_OUT        = f"{OUTPUT_DIR}/04c"
THRESH_DIR   = f"{C_OUT}/threshold_optimization"
SIG_DIR      = f"{C_OUT}/statistical_significance"
WRITEUP_DIR  = f"{C_OUT}/error_analysis_writeup"
EFF_DIR      = f"{C_OUT}/efficiency"
PUB_TAB_DIR  = f"{C_OUT}/publication_tables"
PUB_FIG_DIR  = f"{C_OUT}/publication_figures"

for d in [C_OUT, THRESH_DIR, SIG_DIR, WRITEUP_DIR, EFF_DIR, PUB_TAB_DIR, PUB_FIG_DIR]:
    for sub in ["", "/figures", "/tables"]:
        os.makedirs(d + sub, exist_ok=True)

ALL_MODELS = ["LogReg", "SVM", "MARBERT", "AraBERT", "DziriBERT", "XLM-R"]
CLASSICAL_MODELS = ["LogReg", "SVM"]
TRANSFORMER_MODELS_LIST = ["MARBERT", "AraBERT", "DziriBERT", "XLM-R"]
LABEL_NAMES = ["benign", "adversarial"]

print("Config loaded.")


## 3. Part 1 — Load Every Required Artifact

Fails loudly (`FileNotFoundError`) if a required `05a`/`04b` artifact is missing, naming the exact
path and which upstream cell produces it.

In [ ]:
def _require(path, produced_by):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"04c cannot proceed -- missing required artifact:\n  {path}\n"
            f"  (produced by: {produced_by})\n"
            f"Re-run that step in 04a/04b, then re-run this cell."
        )
    return path


def load_pkl(exp_key):
    path = f"{PREDS_DIR}/{exp_key}_results.pkl"
    if not os.path.exists(path):
        return None
    with open(path, "rb") as f:
        return pickle.load(f)


saved_osact    = load_pkl("osact")
saved_unified  = load_pkl("unified")
saved_transfer = load_pkl("aaac_transfer")
saved_aaac_ref = load_pkl("aaac_reference")

if saved_unified is None:
    raise FileNotFoundError(
        "04c cannot proceed -- Unified test predictions ('unified_results.pkl') are missing. "
        "This is the primary basis for Experiments 6, 7, 8. Re-run 04a Experiment 3 first."
    )

cross_df = pd.read_csv(_require(
    f"{CROSS_DIR}/tables/experiment4_cross_dataset_full.csv", "04b Part 2 (Experiment 4), cell producing cross_df"))
lang_df = pd.read_csv(_require(
    f"{LANG_DIR}/tables/experiment5_language_breakdown.csv", "04b Part 3 (Experiment 5)"))
ablation_df = pd.read_csv(_require(
    f"{ABLATION_DIR}/tables/experiment9_ablation_full.csv", "04b Part 4 (Experiment 9)"))
transfer_df = pd.read_csv(_require(
    f"{SUMMARY_DIR}/tables/dataset_transfer_stats.csv", "04b Part 6 (dataset transfer stats)"))
cos_df = pd.read_csv(_require(
    f"{SUMMARY_DIR}/tables/dataset_transfer_similarity.csv", "04b Part 6 (dataset transfer stats)"))
research_report_04b = pd.read_csv(_require(
    f"{B_OUT}/04b_research_report.csv", "04b Part 9 (research report)"))

print("Loaded 04a result dicts:", {k: (list(v.keys()) if v else None) for k, v in
      {"osact": saved_osact, "unified": saved_unified,
       "aaac_transfer": saved_transfer, "aaac_reference": saved_aaac_ref}.items()})
print("Loaded 04b tables:", {n: df.shape for n, df in
      {"cross_df": cross_df, "lang_df": lang_df, "ablation_df": ablation_df,
       "transfer_df": transfer_df, "cos_df": cos_df,
       "research_report_04b": research_report_04b}.items()})


## 4. Part 2 — Experiment 6: Threshold Optimization

Every model so far has been scored at the implicit default decision threshold of 0.5. This
experiment sweeps the threshold on the **Unified test set** (the primary, most complete deployment
scenario) for every model that has `y_proba`, and reports:

- the threshold that maximizes Macro F1 ("optimal"),
- the resulting Macro F1 / precision / recall / FPR / FNR at that threshold vs. at 0.5,
- full precision-recall-vs-threshold curves (saved as figures) for transparency.

This is descriptive/diagnostic only — no model is retrained, and the benchmark's headline numbers
(computed at 0.5) are not changed by this exercise; it quantifies how much room there is if a
deployment wanted to trade precision for recall (e.g. safety-critical settings favoring lower FNR).

In [ ]:
def sweep_thresholds(y_true, y_proba, n_steps=197):
    thresholds = np.linspace(0.01, 0.99, n_steps)
    rows = []
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
        fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
        rows.append({"threshold": t, "macro_precision": p, "macro_recall": r, "macro_f1": f1,
                      "fpr": fpr, "fnr": fnr,
                      "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                      "mcc": matthews_corrcoef(y_true, y_pred)})
    return pd.DataFrame(rows)


threshold_results = {}
threshold_summary_rows = []

for model_key in ALL_MODELS:
    entry = saved_unified.get(model_key)
    if entry is None or entry.get("y_proba") is None:
        print(f"[skip] {model_key}: no y_proba available in saved_unified results.")
        continue

    y_true = entry["y_true"]
    y_proba = entry["y_proba"]
    sweep = sweep_thresholds(y_true, y_proba)
    threshold_results[model_key] = sweep
    sweep.to_csv(f"{THRESH_DIR}/tables/{model_key}_threshold_sweep.csv", index=False)

    best_row = sweep.loc[sweep["macro_f1"].idxmax()]
    default_row = sweep.iloc[(sweep["threshold"] - 0.5).abs().idxmin()]

    threshold_summary_rows.append({
        "model": model_key,
        "default_threshold": 0.5,
        "default_macro_f1": default_row["macro_f1"],
        "default_fpr": default_row["fpr"], "default_fnr": default_row["fnr"],
        "optimal_threshold": round(float(best_row["threshold"]), 3),
        "optimal_macro_f1": best_row["macro_f1"],
        "optimal_fpr": best_row["fpr"], "optimal_fnr": best_row["fnr"],
        "macro_f1_gain_vs_default": best_row["macro_f1"] - default_row["macro_f1"],
    })

threshold_summary = pd.DataFrame(threshold_summary_rows).sort_values(
    "macro_f1_gain_vs_default", ascending=False)
threshold_summary.to_csv(f"{THRESH_DIR}/tables/experiment6_threshold_summary.csv", index=False)
display(threshold_summary)


In [ ]:
# Precision / recall / F1 vs threshold curves, one panel per model
n_models = len(threshold_results)
fig, axes = plt.subplots(1, n_models, figsize=(4.2 * n_models, 3.6), sharey=True)
if n_models == 1:
    axes = [axes]
for ax, (model_key, sweep) in zip(axes, threshold_results.items()):
    ax.plot(sweep["threshold"], sweep["macro_precision"], label="Precision", lw=1.4)
    ax.plot(sweep["threshold"], sweep["macro_recall"], label="Recall", lw=1.4)
    ax.plot(sweep["threshold"], sweep["macro_f1"], label="Macro F1", lw=2.0, color="black")
    opt_t = threshold_summary.loc[threshold_summary["model"] == model_key, "optimal_threshold"].values[0]
    ax.axvline(opt_t, color="red", ls="--", lw=1, label=f"optimal={opt_t:.2f}")
    ax.axvline(0.5, color="gray", ls=":", lw=1, label="default=0.50")
    ax.set_title(model_key, fontsize=10)
    ax.set_xlabel("Decision threshold")
axes[0].set_ylabel("Score")
axes[0].legend(fontsize=7, loc="lower center")
plt.tight_layout()
fig.savefig(f"{THRESH_DIR}/figures/experiment6_threshold_curves.png", dpi=300)
fig.savefig(f"{THRESH_DIR}/figures/experiment6_threshold_curves.pdf")
plt.show()


In [ ]:
# Precision-Recall curves (threshold-free view), all models overlaid
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for model_key in ALL_MODELS:
    entry = saved_unified.get(model_key)
    if entry is None or entry.get("y_proba") is None:
        continue
    p, r, _ = precision_recall_curve(entry["y_true"], entry["y_proba"])
    pr_auc = entry.get("pr_auc")
    label = f"{model_key} (AP={pr_auc:.3f})" if pr_auc is not None else model_key
    ax.plot(r, p, lw=1.6, label=label)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Unified test set")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f"{THRESH_DIR}/figures/experiment6_pr_curves_overlay.png", dpi=300)
fig.savefig(f"{THRESH_DIR}/figures/experiment6_pr_curves_overlay.pdf")
plt.show()


## 5. Part 3 — Experiment 7: Statistical Significance

Two complements to the point estimates reported elsewhere:

1. **Pairwise McNemar tests** between all 6 models on the **same** Unified test set (paired
   design — same items, same gold labels — is exactly what McNemar requires), with
   **Holm-Bonferroni correction** across the C(6,2)=15 pairwise comparisons to control the
   family-wise error rate.
2. **Multi-seed variance** for every transformer already computed by `05a` (`per_seed`), surfaced
   here as one comparison table — this is what showed XLM-R's seed-instability finding in the
   session summary.

Bootstrap 95% CIs for Macro F1 (`f1_ci_low`/`f1_ci_high`) were already computed per-model in
`05a`'s `compute_all_metrics` and are reported alongside, not recomputed.

In [ ]:
def run_mcnemar(y_true, y_pred_a, y_pred_b):
    y_true = np.array(y_true)
    correct_a = (np.array(y_pred_a) == y_true)
    correct_b = (np.array(y_pred_b) == y_true)
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    table = [[int(np.sum(correct_a & correct_b)), b], [c, int(np.sum(~correct_a & ~correct_b))]]
    result = mcnemar(table, exact=(b + c < 25))
    return result.pvalue, b, c


available_models = [mk for mk in ALL_MODELS if mk in saved_unified and "y_pred" in saved_unified[mk]]
n_expected = len(saved_unified[available_models[0]]["y_true"])
for mk in available_models:
    if len(saved_unified[mk]["y_true"]) != n_expected:
        raise ValueError(
            f"Model '{mk}' Unified y_true length ({len(saved_unified[mk]['y_true'])}) does not "
            f"match the others ({n_expected}) -- these predictions were not scored on the same "
            f"test set and cannot be paired for McNemar. Re-run 04a Experiment 3 for all models "
            f"against the current layer1_test.csv."
        )

pair_rows = []
for i, mk_a in enumerate(available_models):
    for mk_b in available_models[i + 1:]:
        pval, b, c = run_mcnemar(saved_unified[mk_a]["y_true"],
                                  saved_unified[mk_a]["y_pred"],
                                  saved_unified[mk_b]["y_pred"])
        pair_rows.append({"model_a": mk_a, "model_b": mk_b, "b_a_only_correct": b,
                           "c_b_only_correct": c, "p_value_raw": pval})

pairwise_df = pd.DataFrame(pair_rows)
_, pvals_corrected, _, _ = multipletests(pairwise_df["p_value_raw"], alpha=0.05, method="holm")
pairwise_df["p_value_holm"] = pvals_corrected
pairwise_df["significant_at_0.05_holm"] = pairwise_df["p_value_holm"] < 0.05
pairwise_df = pairwise_df.sort_values("p_value_holm")
pairwise_df.to_csv(f"{SIG_DIR}/tables/experiment7_pairwise_mcnemar.csv", index=False)
display(pairwise_df)


In [ ]:
# Bootstrap CI + accuracy summary, one row per (experiment, model)
ci_rows = []
for exp_name, saved in [("osact", saved_osact), ("unified", saved_unified),
                         ("aaac_transfer", saved_transfer), ("aaac_reference", saved_aaac_ref)]:
    if not saved:
        continue
    for model_key, m in saved.items():
        if "f1_ci_low" not in m:
            continue
        ci_rows.append({
            "experiment": exp_name, "model": model_key,
            "macro_f1": m.get("macro_f1"),
            "f1_ci_low": m.get("f1_ci_low"), "f1_ci_high": m.get("f1_ci_high"),
            "ci_width": (m.get("f1_ci_high") - m.get("f1_ci_low"))
                        if m.get("f1_ci_high") is not None and m.get("f1_ci_low") is not None else None,
        })
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(f"{SIG_DIR}/tables/experiment7_bootstrap_ci_summary.csv", index=False)
display(ci_df)


In [ ]:
# Multi-seed variance summary (transformers only -- classical models are deterministic given a seed)
seed_var_rows = []
for exp_name, saved in [("osact", saved_osact), ("unified", saved_unified), ("aaac_transfer", saved_transfer)]:
    if not saved:
        continue
    for model_key in TRANSFORMER_MODELS_LIST:
        m = saved.get(model_key)
        if not m or "per_seed" not in m:
            continue
        seed_var_rows.append({
            "experiment": exp_name, "model": model_key,
            "accuracy_std": m.get("accuracy_std"), "macro_f1_std": m.get("macro_f1_std"),
            "mcc_std": m.get("mcc_std"),
            "macro_f1_mean": m.get("macro_f1_mean"),
            "n_seeds": len(m["per_seed"]),
        })
seed_variance_df = pd.DataFrame(seed_var_rows)
seed_variance_df.to_csv(f"{SIG_DIR}/tables/experiment7_multiseed_variance.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
if len(seed_variance_df):
    plot_df = seed_variance_df.pivot(index="model", columns="experiment", values="macro_f1_std")
    plot_df.plot(kind="bar", ax=ax)
    ax.set_ylabel("Macro F1 std across 3 seeds")
    ax.set_title("Experiment 7 — Seed Stability by Model")
    plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(f"{SIG_DIR}/figures/experiment7_seed_stability.png", dpi=300)
plt.show()
display(seed_variance_df)


## 6. Part 4 — Experiment 8: Error-Analysis Writeup

Quantitative synthesis of `04b`'s Part 5 error-analysis exports (`correct_by_all_models.csv`,
`missed_by_all_models.csv`, per-model `*_false_positives.csv` / `*_false_negatives.csv`) joined
against the language-breakdown table (Experiment 5) to answer: **where do errors concentrate, and
is it consistent across models?** No new inference is run; everything below is aggregation over
already-exported CSVs.

In [ ]:
import glob

error_csvs = {os.path.basename(p).replace(".csv", ""): pd.read_csv(p)
              for p in glob.glob(f"{ERROR_DIR}/*.csv")}
print("Loaded error-analysis CSVs:", list(error_csvs.keys()))

agreement_summary = error_csvs.get("agreement_summary")
missed_by_all = error_csvs.get("missed_by_all_models")
correct_by_all = error_csvs.get("correct_by_all_models")

writeup_rows = []
if agreement_summary is not None:
    display(agreement_summary)
    writeup_rows.append({
        "finding": "cross_model_agreement",
        "detail": agreement_summary.to_dict("records"),
    })


In [ ]:
# Error rate by language_variant and by source, restricted to models with FP/FN exports
fp_fn_frames = []
for model_key in ALL_MODELS:
    for tag in ["false_positives", "false_negatives"]:
        key = f"{model_key}_{tag}"
        if key in error_csvs:
            df = error_csvs[key].copy()
            df["model"] = model_key
            df["error_type"] = tag
            fp_fn_frames.append(df)

if fp_fn_frames:
    all_errors = pd.concat(fp_fn_frames, ignore_index=True)

    error_by_lang = (
        all_errors.groupby(["model", "error_type", "language"], dropna=False)
        .size().rename("n_errors").reset_index()
    )
    error_by_lang.to_csv(f"{WRITEUP_DIR}/tables/errors_by_language_variant.csv", index=False)

    error_by_source = (
        all_errors.groupby(["model", "error_type", "dataset"], dropna=False)
        .size().rename("n_errors").reset_index()
    )
    error_by_source.to_csv(f"{WRITEUP_DIR}/tables/errors_by_source.csv", index=False)

    # Normalize against Experiment 5's per-(domain, model, language_variant) support to get a
    # genuine error RATE, not just a raw count (which is dominated by class sizes).
    unified_lang = lang_df[lang_df["domain"] == "Unified"][["model", "language_variant", "n"]]
    rate_df = error_by_lang.merge(
        unified_lang, left_on=["model", "language"], right_on=["model", "language_variant"], how="left"
    )
    rate_df["error_rate"] = rate_df["n_errors"] / rate_df["n"]
    rate_df = rate_df.drop(columns=["language_variant"]).sort_values("error_rate", ascending=False)
    rate_df.to_csv(f"{WRITEUP_DIR}/tables/error_rate_by_language_variant.csv", index=False)
    display(rate_df.head(15))
else:
    print("[WARN] no per-model FP/FN CSVs found in 04b's error_analysis directory.")
    all_errors, error_by_lang, rate_df = None, None, None


In [ ]:
# Token-length profile of errors vs. correctly-classified rows (a common, easily-verified error driver)
if fp_fn_frames and "token_length" in all_errors.columns:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(data=all_errors, x="model", y="token_length", hue="error_type", ax=ax)
    ax.set_title("Experiment 8 — Token Length of Errors, by Model")
    ax.set_ylabel("Token length")
    plt.xticks(rotation=20)
    plt.tight_layout()
    fig.savefig(f"{WRITEUP_DIR}/figures/experiment8_error_token_length.png", dpi=300)
    plt.show()

    print("Median token length -- errors vs. all Unified test rows:")
    print(all_errors.groupby(["model", "error_type"])["token_length"].median())


In [ ]:
# Auto-generated writeup: rule-based bullet points from the tables above, for direct inclusion
# (with light human editing) in an EMNLP/TDSC error-analysis section.
writeup_lines = ["# Experiment 8 -- Error Analysis Writeup (auto-generated, verify before citing)", ""]

if agreement_summary is not None:
    total = None
    try:
        total = int(agreement_summary["count"].sum())
    except Exception:
        pass
    writeup_lines.append("## Cross-model agreement")
    for _, r in agreement_summary.iterrows():
        writeup_lines.append(f"- **{r['category']}**: {int(r['count'])} rows")
    writeup_lines.append("")

if 'rate_df' in dir() and rate_df is not None and len(rate_df):
    worst = rate_df.sort_values("error_rate", ascending=False).head(5)
    writeup_lines.append("## Highest error-rate (model, language_variant) segments")
    for _, r in worst.iterrows():
        writeup_lines.append(
            f"- **{r['model']}** on **{r['language']}** ({r['error_type']}): "
            f"{r['error_rate']:.1%} error rate ({int(r['n_errors'])}/{int(r['n'])} rows)"
        )
    writeup_lines.append("")

if fp_fn_frames and "token_length" in all_errors.columns:
    med_err = all_errors["token_length"].median()
    writeup_lines.append("## Length effect")
    writeup_lines.append(f"- Median token length among all exported errors: {med_err:.1f} tokens "
                          f"(compare against Unified test's overall token-length distribution from "
                          f"`03`'s Table 6 to state whether errors skew short/long).")
    writeup_lines.append("")

writeup_lines.append("## Caveats")
writeup_lines.append("- This writeup is descriptive/quantitative only. Any claim about *why* a "
                      "specific example was misclassified (sarcasm, code-switching, dialectal slang, "
                      "etc.) requires manual reading of the exported false-positive/false-negative "
                      "CSVs and should be added by hand, not inferred automatically.")

writeup_text = "\n".join(writeup_lines)
with open(f"{WRITEUP_DIR}/experiment8_writeup.md", "w") as f:
    f.write(writeup_text)
print(writeup_text)


## 7. Part 5 — Experiment 10: Efficiency

Consolidates every cost metric already recorded by `04a` (train/inference time, model size,
parameter count) into one comparison table + figures, extending `04b`'s single performance-vs-time
scatter (Unified only) into a full efficiency section across **all three experiments** (OSACT,
Unified, AAAC-transfer) where the underlying artifacts exist.

In [ ]:
eff_rows = []
for exp_name, saved in [("osact", saved_osact), ("unified", saved_unified), ("aaac_transfer", saved_transfer)]:
    if not saved:
        continue
    for model_key, m in saved.items():
        eff_rows.append({
            "experiment": exp_name, "model": model_key,
            "macro_f1": m.get("macro_f1"),
            "train_time_s": m.get("train_time_s"), "infer_time_s": m.get("infer_time_s"),
            "model_mb": m.get("model_mb"), "n_params": m.get("n_params"),
            "gpu_mem_mb": m.get("gpu_mem_mb"),
        })
eff_df = pd.DataFrame(eff_rows)

# Throughput: rows/sec on that experiment's test set (test-set size taken from y_true length)
test_sizes = {}
for exp_name, saved in [("osact", saved_osact), ("unified", saved_unified), ("aaac_transfer", saved_transfer)]:
    if saved:
        for model_key, m in saved.items():
            if "y_true" in m and m["y_true"] is not None:
                test_sizes[(exp_name, model_key)] = len(m["y_true"])

eff_df["test_set_n"] = eff_df.apply(lambda r: test_sizes.get((r["experiment"], r["model"])), axis=1)
eff_df["throughput_rows_per_sec"] = eff_df["test_set_n"] / eff_df["infer_time_s"]

eff_df.to_csv(f"{EFF_DIR}/tables/experiment10_efficiency_full.csv", index=False)
display(eff_df.sort_values(["experiment", "macro_f1"], ascending=[True, False]))


In [ ]:
# Model size and parameter count are per-model architecture properties (constant across
# experiments for transformers; classical models' "n_params" is the TF-IDF vocab size, which
# does vary slightly by training corpus -- both are reported honestly, not conflated).
size_df = eff_df[eff_df["experiment"] == "unified"][["model", "model_mb", "n_params"]].drop_duplicates()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=size_df.sort_values("model_mb"), x="model", y="model_mb", ax=axes[0], color="#4C72B0")
axes[0].set_title("Model size (MB)"); axes[0].set_ylabel("MB")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=size_df.sort_values("n_params"), x="model", y="n_params", ax=axes[1], color="#55A868")
axes[1].set_title("Parameter / feature count"); axes[1].set_ylabel("count")
axes[1].set_yscale("log")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
fig.savefig(f"{EFF_DIR}/figures/experiment10_model_size_params.png", dpi=300)
plt.show()


In [ ]:
# Performance vs. throughput, across all three experiments (extends 04b's Unified-only scatter)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for exp_name, marker in [("osact", "o"), ("unified", "s"), ("aaac_transfer", "^")]:
    sub = eff_df[eff_df["experiment"] == exp_name]
    if not len(sub):
        continue
    sns.scatterplot(data=sub, x="throughput_rows_per_sec", y="macro_f1", s=130, marker=marker,
                     label=exp_name, ax=ax)
    for _, r in sub.iterrows():
        ax.annotate(r["model"], (r["throughput_rows_per_sec"], r["macro_f1"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Throughput (rows/sec, log scale)"); ax.set_ylabel("Macro F1")
ax.set_title("Experiment 10 — Performance vs. Throughput")
plt.tight_layout()
fig.savefig(f"{EFF_DIR}/figures/experiment10_performance_vs_throughput.png", dpi=300)
fig.savefig(f"{EFF_DIR}/figures/experiment10_performance_vs_throughput.pdf")
plt.show()


## 8. Part 6 — Experiment 11: Publication Tables

Camera-ready tables (.csv + .md + .tex), assembled purely from already-computed results —
no new numbers are produced here, only formatting/aggregation for direct paper inclusion.

In [ ]:
def save_all_formats(df, name, float_format="%.4f"):
    df.to_csv(f"{PUB_TAB_DIR}/tables/{name}.csv", index=False)
    with open(f"{PUB_TAB_DIR}/tables/{name}.md", "w") as f:
        f.write(df.to_markdown(index=False, floatfmt=".4f"))
    try:
        with open(f"{PUB_TAB_DIR}/tables/{name}.tex", "w") as f:
            f.write(df.to_latex(index=False, float_format=float_format))
    except Exception as e:
        print(f"[WARN] LaTeX export skipped for {name}: {e}")
    print(f"Saved {name}.{{csv,md,tex}} ({len(df)} rows)")


# --- Table 1: Main results (all 3 experiments x 6 models, mean +- std where available) ---
# NOTE: aaac_reference is loaded verbatim from the PUBLISHED AAAC benchmark's results.json, which
# stores macro_f1_mean/_std but has no per_seed list (it's not retrained anywhere in 04a/04b/04c).
# Without a guard, this function would print "0.9817 +/- 0.0040" for that column exactly like it
# does for genuinely multi-seed columns (OSACT, Unified) -- silently implying 3-seed reproducibility
# for a value that's actually a single published run. Since this table goes straight into
# table1_main_results.tex for the paper, that's a correctness issue, not just a display nuance.
# Fix: force a plain value (no +/-) whenever the source experiment is aaac_reference.
def fmt_mean_std(saved, model_key, metric, force_single=False):
    if not saved or model_key not in saved:
        return None
    m = saved[model_key]
    mean_v, std_v = m.get(f"{metric}_mean"), m.get(f"{metric}_std")
    if mean_v is not None and std_v is not None and not force_single:
        return f"{mean_v:.4f} +/- {std_v:.4f}"
    val = mean_v if mean_v is not None else m.get(metric)
    return f"{val:.4f}" if val is not None else None


main_rows = []
for model_key in ALL_MODELS:
    row = {"model": model_key}
    for exp_name, saved in [("AAAC (reference)", saved_aaac_ref), ("OSACT", saved_osact),
                             ("Unified", saved_unified)]:
        is_reference_exp = (exp_name == "AAAC (reference)")
        row[f"{exp_name}_macro_f1"] = fmt_mean_std(saved, model_key, "macro_f1", force_single=is_reference_exp)
        row[f"{exp_name}_roc_auc"]  = fmt_mean_std(saved, model_key, "roc_auc", force_single=is_reference_exp)
    main_rows.append(row)
main_results_table = pd.DataFrame(main_rows)
save_all_formats(main_results_table, "table1_main_results")
display(main_results_table)
print("Note: 'AAAC (reference)' columns are single values from the published AAAC benchmark "
      "(not retrained here, no per-seed spread available in this pipeline) -- only 'OSACT' and "
      "'Unified' columns are genuine 3-seed mean +/- std.")


In [ ]:
# --- Table 2: Cross-dataset transfer matrix (Macro F1) ---
transfer_matrix = cross_df.pivot_table(index=["source", "target"], columns="model", values="macro_f1").reset_index()
save_all_formats(transfer_matrix, "table2_cross_dataset_transfer")
display(transfer_matrix)


In [ ]:
# --- Table 3: Ablation (AAAC-only vs OSACT-only vs Unified) ---
ablation_cols = ["model"] + [c for c in ablation_df.columns if c.startswith(("AAAC_only_", "OSACT_only_", "Unified_"))
                              and c.endswith("macro_f1")]
ablation_pub = ablation_df[ablation_cols]
save_all_formats(ablation_pub, "table3_ablation_macro_f1")
display(ablation_pub)


In [ ]:
# --- Table 4: Efficiency summary (Unified experiment) ---
eff_pub = eff_df[eff_df["experiment"] == "unified"][
    ["model", "macro_f1", "train_time_s", "infer_time_s", "throughput_rows_per_sec", "model_mb", "n_params"]
].sort_values("macro_f1", ascending=False)
save_all_formats(eff_pub, "table4_efficiency_unified")
display(eff_pub)


In [ ]:
# --- Table 5: Threshold optimization summary ---
save_all_formats(threshold_summary, "table5_threshold_optimization")

# --- Table 6: Statistical significance (pairwise McNemar, Holm-corrected) ---
save_all_formats(pairwise_df, "table6_pairwise_significance")

# --- Table 7: Dataset statistics recap (from 04b's domain-transfer stats) ---
save_all_formats(transfer_df, "table7_dataset_transfer_stats")

print("\nAll publication tables written to:", f"{PUB_TAB_DIR}/tables/")


## 9. Part 7 — Experiment 12: Remaining Publication Figures

Figures not already produced by `04b` (confusion matrices, cross-dataset heatmap, per-language bar
charts, ablation bar chart, radar chart, ROC/PR overlays, single performance-vs-time scatter) or by
Experiment 6/10 above (threshold curves, PR overlay, size/throughput scatter): a consolidated
multi-panel overview figure, and a seed-stability-vs-performance combined figure.

In [ ]:
# Consolidated overview: Macro F1 per model across the three primary experiments, one panel
fig, ax = plt.subplots(figsize=(8, 5))
overview_rows = []
for exp_name, saved in [("AAAC (reference)", saved_aaac_ref), ("OSACT", saved_osact), ("Unified", saved_unified)]:
    if not saved:
        continue
    for model_key in ALL_MODELS:
        m = saved.get(model_key)
        if m is None:
            continue
        overview_rows.append({"experiment": exp_name, "model": model_key, "macro_f1": m.get("macro_f1")})
overview_df = pd.DataFrame(overview_rows)
if len(overview_df):
    pivot = overview_df.pivot(index="model", columns="experiment", values="macro_f1")
    pivot = pivot.reindex(ALL_MODELS)
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("Macro F1")
    ax.set_title("Experiment 12 — Macro F1 Across Experiments, All Models")
    plt.xticks(rotation=20)
plt.tight_layout()
fig.savefig(f"{PUB_FIG_DIR}/figures/experiment12_overview_macro_f1.png", dpi=300)
fig.savefig(f"{PUB_FIG_DIR}/figures/experiment12_overview_macro_f1.pdf")
plt.show()


In [ ]:
# Combined figure: performance (Unified Macro F1) vs. seed stability (Unified Macro F1 std)
fig, ax = plt.subplots(figsize=(7, 5.5))
combo_rows = []
for model_key in ALL_MODELS:
    m = saved_unified.get(model_key, {})
    combo_rows.append({
        "model": model_key,
        "macro_f1": m.get("macro_f1"),
        "macro_f1_std": m.get("macro_f1_std", 0.0),  # classical models: no seed variation -> 0
    })
combo_df = pd.DataFrame(combo_rows)
sns.scatterplot(data=combo_df, x="macro_f1_std", y="macro_f1", s=150, ax=ax)
for _, r in combo_df.iterrows():
    ax.annotate(r["model"], (r["macro_f1_std"], r["macro_f1"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Macro F1 std across 3 seeds (0 for classical models)")
ax.set_ylabel("Macro F1 (Unified test)")
ax.set_title("Experiment 12 — Performance vs. Seed Stability")
plt.tight_layout()
fig.savefig(f"{PUB_FIG_DIR}/figures/experiment12_performance_vs_stability.png", dpi=300)
fig.savefig(f"{PUB_FIG_DIR}/figures/experiment12_performance_vs_stability.pdf")
plt.show()


## 10. Part 8 — Research Report (`04c_research_report.csv`)

Flattens every table produced in this notebook into one long-format CSV, in the same
`experiment / source / target / model / metric / value` shape as `04b_research_report.csv`, so the
three notebooks' reports can be concatenated into a single appendix table.

In [ ]:
report_rows = []

for _, row in threshold_summary.iterrows():
    for metric in ["default_macro_f1", "optimal_macro_f1", "macro_f1_gain_vs_default", "optimal_threshold"]:
        report_rows.append({
            "experiment": "Experiment 6 - Threshold Optimization",
            "source": "Unified", "target": "Unified", "model": row["model"],
            "metric": metric, "value": row[metric],
        })

for _, row in pairwise_df.iterrows():
    report_rows.append({
        "experiment": "Experiment 7 - Statistical Significance",
        "source": row["model_a"], "target": row["model_b"], "model": f"{row['model_a']}_vs_{row['model_b']}",
        "metric": "p_value_holm", "value": row["p_value_holm"],
    })

for _, row in seed_variance_df.iterrows():
    report_rows.append({
        "experiment": "Experiment 7 - Multi-seed Variance",
        "source": row["experiment"], "target": row["experiment"], "model": row["model"],
        "metric": "macro_f1_std", "value": row["macro_f1_std"],
    })

for _, row in eff_df.iterrows():
    for metric in ["train_time_s", "infer_time_s", "throughput_rows_per_sec", "model_mb"]:
        report_rows.append({
            "experiment": "Experiment 10 - Efficiency",
            "source": row["experiment"], "target": row["experiment"], "model": row["model"],
            "metric": metric, "value": row[metric],
        })

report_df_04c = pd.DataFrame(report_rows)
report_df_04c.to_csv(f"{C_OUT}/04c_research_report.csv", index=False)
print("Saved research report:", f"{C_OUT}/04c_research_report.csv", "-", len(report_df_04c), "rows")
display(report_df_04c.head(20))


In [ ]:
# Combined appendix table across all three notebooks' research reports
combined_report = pd.concat([
    research_report_04b.assign(notebook="04b"),
    report_df_04c.assign(notebook="04c"),
], ignore_index=True)
combined_report.to_csv(f"{C_OUT}/combined_04b_04c_research_report.csv", index=False)
print("Saved combined report:", f"{C_OUT}/combined_04b_04c_research_report.csv", "-", len(combined_report), "rows")
